In [35]:
from io import BytesIO
import sys
!{sys.executable} -m pip install fsspec s3fs oci ocifs 
!{sys.executable} -m pip install pandas numpy==1.24.3
!{sys.executable} -m pip uninstall pyarrow -y
!{sys.executable} -m pip install pyarrow==12.0.1

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Found existing installation: pyarrow 12.0.1
Uninstalling pyarrow-12.0.1:
  Successfully uninstalled pyarrow-12.0.1
Defaulting to user installation because normal site-packages is not writeable
  Using cached pyarrow-12.0.1-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (39.0 MB)


In [36]:
import oci
import ocifs
import pandas as pd
import sys
import os
import pyarrow

In [37]:
#Buckets e nomes de saída nuvem = "oci://"
pasta_in = 'base_score_bureau_movel_full/'
pasta_out = 'Feature_store/'
namespace = "@grxzqsiaote6/"
bucket_trusted = f"oci://TRUSTED{namespace}{pasta_in}"
bucket_feature_store = f"oci://BOOKS_VARIAVEIS{namespace}{pasta_out}"

In [38]:
#Sugestao do estagiario
#NAMESPACE = "@grxzqsiaote6"
#pasta_in = "base_score_bureau_movel_full/"
#pasta_out = "Feature_store/"
#bucket_trusted = f"oci://TRUSTED{NAMESPACE}/{pasta_in}"
#bucket_feature_store = f"oci://BOOKS_VARIAVEIS{NAMESPACE}/{pasta_out}"

In [39]:
from ocifs import OCIFileSystem

fs = OCIFileSystem(config="~/.oci/config")

#files = fs.ls("oci://TRUSTED@grxzqsiaote6/base_score_bureau_movel_full/")
files = fs.ls(bucket_trusted)

print(files)

['TRUSTED@grxzqsiaote6/base_score_bureau_movel_full/_SUCCESS', 'TRUSTED@grxzqsiaote6/base_score_bureau_movel_full/ts_proc_partition=20260309213950']


In [40]:
df_bureau = pd.read_parquet(
    bucket_trusted, #"oci://TRUSTED@grxzqsiaote6/base_score_bureau_movel_full/",
    storage_options={"config": "~/.oci/config"}
)

df_bureau.head()

,ts_proc,Ano,Mes,FLAG_INSTALACAO,ProductDescription,ProductMigration,SCORE_01,SCORE_02,FPD,NUM_CPF,ts_proc_partition,SAFRA
0,20260309213950,2024,10,True,CMV,Aquisição,2.0,1.0,1.0,ZZZZZZZX7T9,20260309213950,202410
1,20260309213950,2024,10,False,CMV,None,562.0,559.0,NaN,ZZZZZZZ8TZ8,20260309213950,202410
2,20260309213950,2024,10,False,CMV,None,585.0,559.0,NaN,ZZZZZZW9XWN,20260309213950,202410
3,20260309213950,2024,10,True,CMV,PRE,562.0,636.0,0.0,ZZZZZX7XWY8,20260309213950,202410
4,20260309213950,2024,10,True,CMV,Aquisição,538.0,570.0,1.0,ZZZZZX8TTUZ,20260309213950,202410


In [41]:
df_bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3795310 entries, 0 to 3795309
Data columns (total 12 columns):
 #   Column              Dtype   
---  ------              -----   
 0   ts_proc             object  
 1   Ano                 int32   
 2   Mes                 int32   
 3   FLAG_INSTALACAO     bool    
 4   ProductDescription  object  
 5   ProductMigration    object  
 6   SCORE_01            float32 
 7   SCORE_02            float32 
 8   FPD                 float64 
 9   NUM_CPF             object  
 10  ts_proc_partition   category
 11  SAFRA               category
dtypes: bool(1), category(2), float32(2), float64(1), int32(2), object(4)
memory usage: 213.6+ MB


In [42]:
# Ajuste de tipos para otimizar memória e processamento

df_bureau['Ano'] = pd.to_numeric(df_bureau['Ano'], errors='coerce').astype('Int32')
df_bureau['Mes'] = pd.to_numeric(df_bureau['Mes'], errors='coerce').astype('Int8')

df_bureau['FLAG_INSTALACAO'] = df_bureau['FLAG_INSTALACAO'].astype('boolean')

df_bureau['ProductDescription'] = df_bureau['ProductDescription'].astype('string')  # PROD
df_bureau['ProductMigration'] = df_bureau['ProductMigration'].astype('string')  # flag_mig2

df_bureau['SCORE_01'] = pd.to_numeric(df_bureau['SCORE_01'], errors='coerce').astype('float32')
df_bureau['SCORE_02'] = pd.to_numeric(df_bureau['SCORE_02'], errors='coerce').astype('float32')

df_bureau['FPD'] = pd.to_numeric(df_bureau['FPD'], errors='coerce').astype('Int32')

df_bureau['NUM_CPF'] = df_bureau['NUM_CPF'].astype('string')

df_bureau['SAFRA'] = pd.to_numeric(df_bureau['SAFRA'], errors='coerce').astype('Int32')

In [43]:
# Iremos criar as medidas descritas acima
df_bureau['SCORE_RATE'] = df_bureau['SCORE_02'] / df_bureau['SCORE_01']
df_bureau['SCORE_AVG'] = (df_bureau['SCORE_01'] + df_bureau['SCORE_02']) / 2
df_bureau['SCORE_DIFF'] = df_bureau['SCORE_02'] - df_bureau['SCORE_01']
df_bureau['SCORE_MIN'] = df_bureau[['SCORE_01', 'SCORE_02']].min(axis=1)

In [44]:
df_bureau.head(5)

,ts_proc,Ano,Mes,FLAG_INSTALACAO,ProductDescription,ProductMigration,SCORE_01,SCORE_02,FPD,NUM_CPF,ts_proc_partition,SAFRA,SCORE_RATE,SCORE_AVG,SCORE_DIFF,SCORE_MIN
0,20260309213950,2024,10,True,CMV,Aquisição,2.0,1.0,1,ZZZZZZZX7T9,20260309213950,202410,0.500000,1.5,-1.0,1.0
1,20260309213950,2024,10,False,CMV,<NA>,562.0,559.0,<NA>,ZZZZZZZ8TZ8,20260309213950,202410,0.994662,560.5,-3.0,559.0
2,20260309213950,2024,10,False,CMV,<NA>,585.0,559.0,<NA>,ZZZZZZW9XWN,20260309213950,202410,0.955556,572.0,-26.0,559.0
3,20260309213950,2024,10,True,CMV,PRE,562.0,636.0,0,ZZZZZX7XWY8,20260309213950,202410,1.131673,599.0,74.0,562.0
4,20260309213950,2024,10,True,CMV,Aquisição,538.0,570.0,1,ZZZZZX8TTUZ,20260309213950,202410,1.059480,554.0,32.0,538.0


In [47]:
#Adicione storage_options no to_parquet.
df_bureau.to_parquet(
    f"{bucket_feature_store}book_variaveis_01.parquet",
    engine="pyarrow",
#    partition_cols=["SAFRA"],
    index=False,
    storage_options={"config": "~/.oci/config"}
)

In [46]:
# Código para salvar particionado
#df_bureau.to_parquet(
#    bucket_feature_store,
#    engine="pyarrow",
#    compression="snappy",
#    partition_cols=["Ano", "Mes"],
#    index=False,
#    storage_options={"config": "~/.oci/config"}
#)